In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mensimulasikan model Ising ditendang dengan fungsi TEM

import TutorialFeedback from '@site/src/components/TutorialFeedback';



Kaedah Tensor-network Error Mitigation (TEM) daripada Algorithmiq ialah algoritma hibrid kuantum-klasik yang direka untuk melakukan mitigasi hingar sepenuhnya dalam peringkat post-processing klasik. Dengan TEM, pengguna boleh mengira nilai jangkaan pemboleh ukur, mengurangkan ralat yang disebabkan hingar yang tidak dapat dielakkan pada perkakasan kuantum dengan ketepatan dan kecekapan kos yang lebih tinggi, menjadikannya pilihan yang sangat menarik bagi penyelidik kuantum dan pengamal industri.

Tutorial ini menunjukkan bagaimana TEM boleh mendapatkan keputusan yang bermakna untuk dinamik sistem kuantum, yang tidak boleh diakses tanpa mitigasi ralat dan yang memerlukan sumber daya kuantum yang jauh lebih banyak jika kaedah mitigasi ralat lain seperti PEC dan ZNE digunakan.

*Anggaran penggunaan: Buku nota ini menggunakan kira-kira 10 minit QPU pada peranti Heron r3. Masa runtime boleh bergantung secara ketara pada peranti yang dipilih. Anggaran penggunaan mengikut bahagian boleh didapati di bawah.*

## Menjalankan eksperimen fizik banyak-jasad dengan mitigasi ralat menggunakan fungsi TEM
Tutorial ini berdasarkan rujukan berikut: [L. E. Fischer et al., Nat. Phys. (2026)](https://www.nature.com/articles/s41567-025-03144-9). Rujukan ini membincangkan simulasi sebenar pada perkakasan kuantum dengan hingga 91 qubit. Dalam tutorial ini, kita mencipta semula simulasi yang serupa pada saiz litar yang lebih kecil.

Model Ising ditendang sepadan dengan model Ising biasa:

$$ \hat{H}_{\text{I}} = J \sum_{n=0}^{N-2} \hat{Z}_n \hat{Z}_{n+1} + h \sum_{n=0}^{N-1} \hat{Z}_n $$

kepada mana diterapkan tendangan melintang:

$$ \hat{H}_{K} = b \sum_{n=0}^{N-1} \hat{X}_n $$

Matlamatnya adalah untuk mensimulasikan dinamik suatu keadaan di bawah Hamiltonian Ising ditendang melintang, yang evolusi masanya boleh dilaksanakan oleh unitary Floquet $\hat{U}_{\text{KI}} = e^{-i \hat{H}_K} e^{-i \hat{H}_I}$. Keadaan awal untuk dievolusi ialah yang mana qubit pertama berada dalam keadaan $|+\rangle$, manakala yang lain dipasangkan dan ditetapkan dalam keadaan Bell $(|00\rangle + |11\rangle)/\sqrt{2}$.

Kuantiti yang ingin kita perhatikan ialah fungsi korelasi. [Kertas kerja rujukan](https://www.nature.com/articles/s41567-025-03144-9) membincangkan bagaimana kuantiti ini boleh ditulis semula sebagai operator Pauli $\hat{X}$ pada qubit ke-$n$. Selepas beberapa langkah masa fizikal $t$, kita mengira nilai operator Pauli $\hat{X}_{n=t}$. Bergantung pada parameter sistem, nilai pemboleh ukur ini sama dengan nilai yang boleh dikira secara tepat, atau hanya boleh disimulasikan melalui kaedah anggaran. Khususnya, untuk $|J|=|b|=\pi/4$ ia sama dengan $[\cos(2h)]^t$, yang merupakan nilai yang akan kita gunakan untuk menanda aras keputusan tutorial ini. Tambahan pula, pada langkah masa tertentu $t$, $\langle\hat{X}_{n\neq t}\rangle$ adalah sifar. Untuk butiran mendapatkan nilai-nilai ini, dan untuk perbandingan dengan keputusan simulasi klasik anggaran di luar parameter ini, lihat [L. E. Fischer et al., Nat. Phys. (2026)](https://www.nature.com/articles/s41567-025-03144-9).

TEM berfungsi dengan terlebih dahulu mencirikan hingar untuk setiap lapisan unik gate dua qubit dalam litar, serta mencirikan ralat bacaan. Kemudian, litar dilaksanakan pada mesin kuantum. Akhirnya, mitigasi ralat rangkaian tensor dilakukan pada sumber daya klasik di IBM Cloud&reg; dan nilai yang dimitigasi dikembalikan. Dalam contoh ini, litar mempunyai dua lapisan unik untuk dicirikan.

## Persediaan
Sebagai prasyarat, pastikan kebergantungan yang diperlukan dipasang.

In [ ]:
%pip install numpy matplotlib qiskit qiskit-ibm-catalog qiskit-ibm-runtime pylatexenc qiskit_qasm3_import

In [1]:
import os
from matplotlib import pyplot as plt
import numpy as np

from qiskit.quantum_info import SparsePauliOp
from qiskit.qasm3 import load

from qiskit_ibm_catalog import QiskitFunctionsCatalog

## Mitigasi ralat dengan TEM
Kami menyediakan di sini litar yang melaksanakan model Ising ditendang yang diterangkan di atas. Litar disediakan seperti berikut. Pertama, ada fasa penyediaan keadaan, di mana qubit pertama berada dalam keadaan $|+\rangle$, manakala yang lain dalam pasangan Bell $(|00\rangle + |11\rangle)/\sqrt{2}$. Ini diikuti oleh struktur bata yang melaksanakan evolusi unitary $\hat{U}_{\text{KI}}$. Bilangan langkah masa fizikal sepadan dengan $t/2$ lapisan litar. Kod berikut memuat turun dua fail QASM yang diperlukan untuk tutorial ini.

In [ ]:
# Download required QASM files
import urllib

urllib.request.urlretrieve(
    "https://ibm.box.com/shared/static/swy5jtq309b0xpzluzlmsmj908yphes8.qasm",
    "ki_30q.qasm",
)
urllib.request.urlretrieve(
    "https://ibm.box.com/shared/static/et3gkodonw6gsp2trs43lzaozrdtiu7s.qasm",
    "ki_12q.qasm",
)

Kita boleh menggambarkan versi kecil litar, dengan 12 qubit dan enam langkah masa:

In [3]:
# Parameters of the kicked Ising model
h = 0.0
num_qubits = 12
t_steps = 6

# Load the circuit for the kicked Ising model
small_circuit = load("ki_12q.qasm")

# Draw the circuit
small_circuit.draw("mpl", scale=0.25, fold=-1)

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/381a4e25-bc9c-47d0-b9f1-172eb5516484-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/381a4e25-bc9c-47d0-b9f1-172eb5516484-0.avif)

Seterusnya, bina pemboleh ukur, $\hat{X}_{n=t}$. Ia dibina sebagai rentetan Pauli mudah dengan susunan yang sepadan dengan yang digunakan oleh Qiskit:

In [3]:
def xt_observable(n_qubits, t_steps):
    pauli_str = "".join(["I" * t_steps, "X", "I" * (n_qubits - t_steps - 1)])
    pauli_str = pauli_str[::-1]  # Reverse the string to match qiskit order
    return SparsePauliOp(data=pauli_str, coeffs=1.0)

Dalam contoh 12 qubit kecil kita, pemboleh ukur kelihatan seperti ini:

In [4]:
# Build the observable for the kicked Ising model
small_observable = xt_observable(n_qubits=12, t_steps=6)
print(small_observable)

SparsePauliOp(['IIIIIXIIIIII'],
              coeffs=[1.+0.j])


Qiskit Functions use PUBs as the way to collect the inputs. In our case, let's consider a single circuit and observable as our PUB:

In [ ]:
# Collect the input PUBs, in this case composed of a
# single circuit and observable
pubs = [(small_circuit, [small_observable])]

Qiskit Functions menggunakan PUB sebagai cara untuk mengumpul input. Dalam kes kita, mari pertimbangkan satu litar dan pemboleh ukur sebagai PUB kita:

In [6]:
# Set IBM Quantum credentials and backend configuration
personal_token = os.environ.get(
    "QISKIT_IBM_TOKEN", "<API-KEY>"
)  # Replace with your personal token or set the environment variable
channel = "ibm_quantum_platform"
crn = "your_crn"  # Replace with the Cloud Resource Name (CRN)

# Select the QPU backend
backend_name = "ibm_qpu_name"  # Replace with your desired backend's name

Seterusnya, kita mendapat akses kepada fungsi TEM. Kita mula-mula menyediakan pengesahan yang diperlukan ke IBM Cloud dan memilih Backend daripada peranti yang tersedia. Token, Backend yang tersedia, dan nama sumber daya awan yang berkaitan (CRN) boleh diperoleh dengan masuk ke akaun kamu di [papan pemuka IBM Quantum Platform](https://quantum.cloud.ibm.com/).

In [8]:
# Load the TEM function from the Qiskit Functions Catalog
catalog = QiskitFunctionsCatalog(
    channel=channel,
    token=personal_token,
    instance=crn,
)
tem = catalog.load("algorithmiq/tem")

Muatkan fungsi TEM daripada [Katalog Qiskit Functions](https://quantum.cloud.ibm.com/functions):

In [9]:
tem_job = tem.run(pubs=pubs, backend_name=backend_name)

Kita kini boleh menjalankan eksperimen pada litar Ising ditendang dengan mitigasi ralat yang disediakan oleh TEM. Menggunakan tetapan lalai, TEM boleh dijalankan dengan cara mudah dengan jangkaan runtime QPU sekitar 2.5 minit, bergantung pada QPU:

In [10]:
print(tem_job.status())

QUEUED


Dengan pilihan lalai, fungsi TEM menjalankan tiga kerja pada komputer kuantum: pembelajaran hingar, mitigasi bacaan, dan persampelan litar. Bilangan shots yang digunakan oleh setiap satu boleh diubah dalam pilihan yang dihantar ke fungsi. Secara lalai, parameter ini ditetapkan untuk mencapai ketepatan 0.05 dalam nilai jangkaan yang dimitigasi. Kamu boleh menyemak status kerja kamu di [papan pemuka IBM Quantum Platform](https://quantum.cloud.ibm.com/) atau dengan:

In [11]:
# Get the results of the TEM job
tem_results = tem_job.result()[
    0
]  # Get the first and only result from the job
tem_evs = tem_results.data.evs[0]
tem_std = tem_results.data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")

TEM Result: 1.031 ± 0.046


We can also check how much quantum runtime was used for each call at [IBM Quantum Platform](https://quantum.cloud.ibm.com), or by inspecting the result metadata from the Python code.

In [12]:
# Get the TEM job runtime
tem_runtime = tem_job.result().metadata["resource_usage"][
    "RUNNING: EXECUTING_QPU"
]["QPU_TIME"]

print(f"TEM Runtime: {tem_runtime} seconds")

TEM Runtime: 155.0 seconds


Apabila statusnya `DONE`, kita boleh menyemak keputusan mentah dan yang dimitigasi. `tem_evs` yang ditakrifkan di bawah ialah nilai jangkaan pemboleh ukur yang diminta, dalam kes ini hanya satu pemboleh ukur, $\langle \hat X_{n=t}\rangle$, dan `tem_std` ialah sisihan piawai yang berkaitan.

In [ ]:
options = {
    "default_shots": 10_000,
    "tem_max_bond_dimension": 512,
    "tem_compression_cutoff": 1e-16,
    # This option helps optimizing the measurement
    # stage since the observable is strongly biased
    # toward the X operator for all the qubits.
    "compute_shadows_bias_from_observable": True,
    # set to True to keep experiment results private,
    # recommended for confidential circuits
    "private": False,
}

Custom options for the noise learner can also be passed. They follow the definitions used in the Qiskit Runtime [`NoiseLearnerOptions`](/docs/api/qiskit-ibm-runtime/options-noise-learner-options):

In [29]:
nl_options = {
    "num_randomizations": 32,
    "max_layers_to_learn": 2,
    "shots_per_randomization": 128,
    "layer_pair_depths": [0, 1, 2, 4, 16, 32],
}

# add noise learning options to the overall options
options |= nl_options

Kita juga boleh menyemak berapa banyak runtime kuantum yang digunakan untuk setiap panggilan di [IBM Quantum Platform](https://quantum.cloud.ibm.com), atau dengan memeriksa metadata keputusan daripada kod Python.

In [30]:
tem_job_custom = tem.run(
    pubs=pubs, backend_name=backend_name, options=options
)

If the job is not set as private, we can recover the result at a later point. To do so, save the job ID printed here and use `tem_job_custom = catalog.get_job_by_id("your-job-id")`.

In [31]:
job_id = tem_job_custom.job_id
print(f"Job ID: {job_id}")

Job ID: 1ba10094-a541-457a-9287-dbd49306d12d


In [33]:
results_custom = tem_job_custom.result()
tem_evs = results_custom[0].data.evs[0]
tem_std = results_custom[0].data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")

TEM Result: 0.956 ± 0.018


Pilihan tersuai juga boleh dihantar kepada pembelajaran hingar. Definisi yang digunakan mengikut yang terdapat dalam Qiskit Runtime [`NoiseLearnerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-noise-learner-options):

In [34]:
metadata_custom = results_custom[0].metadata

unmitigated_evs = metadata_custom["evs_non_mitigated"][0]
unmitigated_stds = metadata_custom["stds_non_mitigated"][0]
print(f"Unmitigated Result: {unmitigated_evs:.3f} ± {unmitigated_stds:.3f}")

# Exact result for the kicked Ising model from the reference paper
exact_evs = np.cos(2 * h) ** t_steps
print("Exact Result:", exact_evs)

Unmitigated Result: 0.894 ± 0.015
Exact Result: 1.0


In [35]:
# Plot comparing the different expectation values
plt.bar(
    ["Unmitigated", "TEM"],
    [unmitigated_evs, tem_evs],
    yerr=[unmitigated_stds, tem_std],
    color=["grey", "c"],
)
plt.hlines(y=exact_evs, xmin=-0.5, xmax=1.5, colors="r", linestyles="dashed")
plt.ylabel("Expectation Value")
plt.ylim(0, 1.1)
plt.show()

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/c3a2168d-98df-491e-a1f8-05de5684ab96-0.avif" alt="Output of the previous code cell" />

Jika kerja tidak ditetapkan sebagai peribadi, kita boleh mendapatkan semula keputusan kemudian. Untuk melakukan itu, simpan ID kerja yang dicetak di sini dan gunakan `tem_job_custom = catalog.get_job_by_id("your-job-id")`.

In [27]:
# Get the metadata of the TEM job
job_metadata = results_custom.metadata

# Get the runtime of the TEM job
qpu_runtime = job_metadata["resource_usage"]["RUNNING: EXECUTING_QPU"][
    "QPU_TIME"
]
classical_runtime = (
    job_metadata["resource_usage"]["RUNNING: OPTIMIZING_FOR_HARDWARE"][
        "CPU_TIME"
    ]
    + job_metadata["resource_usage"]["RUNNING: POST_PROCESSING"]["CPU_TIME"]
)

print(f"QPU Runtime: {qpu_runtime} seconds")
print(f"Classical Runtime: {classical_runtime} seconds")

QPU Runtime: 342.0 seconds
Classical Runtime: 107.632604 seconds


## Scale TEM to large circuits

Large circuits can, in principle, be run with the TEM function. However, it is important to be aware of the of the limitations of the classical resources, as TEM is executed on IBM Cloud runners with potentially very long run times. For extremely large circuits, contact the TEM support team at [qiskit_ibm@algorithmiq.fi](mailto:qiskit_ibm@algorithmiq.fi).

Here we run an example with a larger, utility-scale-sized 30-qubit circuit, optimizing the TEM parameters for speed rather than accuracy.

In [ ]:
# Kicked Ising model parameters
n_qubits = 30
t_steps = 15
h = 0.0

# Load the circuit for the kicked Ising model
circuit = load("ki_30q.qasm")


# Build the observable for the kicked Ising model
observable = xt_observable(n_qubits=n_qubits, t_steps=t_steps)

# Collect the input PUBs, in this case composed of a
# single circuit and observable
pubs = [(circuit, [observable])]

Let's define some performance-oriented options:

In [49]:
options = {
    "num_randomizations": 32,
    "max_layers_to_learn": 2,
    "shots_per_randomization": 128,
    "layer_pair_depths": [0, 1, 2, 4, 16, 32, 64],
    "default_shots": 5_000,
    "tem_max_bond_dimension": 128,
    "tem_compression_cutoff": 1e-10,
    "compute_shadows_bias_from_observable": True,
    "private": False,
}

Finally, run the experiment, get the result, and visualize it. This will take approximately 3.5 QPU minutes.

In [50]:
tem_job_large = tem.run(pubs=pubs, backend_name=backend_name, options=options)

In [51]:
job_id = tem_job_large.job_id
print(f"Job ID: {job_id}")

Job ID: 9f3f190f-f4b0-4dcb-bb83-5f71f37d0d77


In [53]:
results_large = tem_job_large.result()
tem_evs = results_large[0].data.evs[0]
tem_std = results_large[0].data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")


# Get the metadata of the TEM job
job_metadata = tem_job_large.result().metadata

# Get the runtime of the TEM job
qpu_runtime = job_metadata["resource_usage"]["RUNNING: EXECUTING_QPU"][
    "QPU_TIME"
]
classical_runtime = (
    job_metadata["resource_usage"]["RUNNING: OPTIMIZING_FOR_HARDWARE"][
        "CPU_TIME"
    ]
    + job_metadata["resource_usage"]["RUNNING: POST_PROCESSING"]["CPU_TIME"]
)

print(f"QPU Runtime: {qpu_runtime} seconds")
print(f"Classical Runtime: {classical_runtime} seconds")

TEM Result: 0.794 ± 0.026
QPU Runtime: 203.0 seconds
Classical Runtime: 251.71805499999996 seconds


In [54]:
# Plot comparing the different expectation values
metadata_large = results_large[0].metadata
unmitigated_evs = metadata_large["evs_non_mitigated"][0]
unmitigated_stds = metadata_large["stds_non_mitigated"][0]

exact_evs = np.cos(2 * h) ** t_steps

plt.bar(
    ["Unmitigated", "TEM"],
    [unmitigated_evs, tem_evs],
    yerr=[unmitigated_stds, tem_std],
    color=["grey", "c"],
)
plt.hlines(y=exact_evs, xmin=-0.5, xmax=1.5, colors="r", linestyles="dashed")
plt.ylabel("Expectation Value")
plt.ylim(0, 1.1)
plt.show()

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/24894c44-e399-4b9d-a3ff-38a28ff32ece-0.avif" alt="Output of the previous code cell" />